A company manufactures six products, denoted $P_1, . . . , P_6$. Production is planned on a weekly
basis. Each unit produced generates a contribution margin and consumes three limited resources:
machining time, assembly time, and finishing time.
The unit data are:

| Product | Unit margin | Maximal sales | Machining | Assembly | Finishing |
|---------|-------------|---------------|-----------|----------|-----------|
| $P_1$   | 32          | 80            | 2         | 4        | 1         |
| $P_2$   | 40          | 70            | 3         | 2        | 2         |
| $P_3$   | 38          | 90            | 2         | 3        | 2         |
| $P_4$   | 55          | 60            | 4         | 2        | 3         |
| $P_5$   | 53          | 75            | 3         | 4        | 1         |
| $P_6$   | 60          | 50            | 5         | 3        | 2         |

The available resources for the coming week are:

| Resource | Available hours |
| -------- | --------------- |
| Machining| 520             |
| Assembly | 480             |
| Finishing| 300             |

Production quantities may be considered continuous.

In [1]:
using JuMP
import HiGHS
using LinearAlgebra

U = [32, 40, 38, 55, 53, 60]
MAX = [80, 70, 90, 60, 75, 50]
M = [2, 3, 2, 4, 3, 5]
A = [4, 2, 3, 2, 4, 3]
F = [1, 2, 2, 3, 1, 2]

N = 6

AM, AA, AF = 520, 480, 300

model = Model(HiGHS.Optimizer)

A JuMP Model
├ solver: HiGHS
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

# Linear production model

Initially assume that producing a product carries no fixed cost and that products can be selected
independently: any quantity between zero and the maximum sales quantity may be produced

Q1. Define appropriate decision variables.

In [2]:
@variable(model, 0 <= Q[i = 1:N] <= MAX[i])

6-element Vector{VariableRef}:
 Q[1]
 Q[2]
 Q[3]
 Q[4]
 Q[5]
 Q[6]

Q2. Formulate a linear optimization problem that maximizes the weekly contribution margin.

$$\begin{align*}
\max_{Q} \quad & U^T Q \\
\text{s.t.} \quad 0 \le & Q \le MAX \\
& M \cdot Q \le AM \\
& A \cdot Q \le AA \\
& F \cdot Q \le AF 
\end{align*}$$

In [3]:
@objective(model, Max, U⋅Q)
@constraint(model, cM, M ⋅ Q <= AM)
@constraint(model, cA, A ⋅ Q <= AA)
@constraint(model, cF, F ⋅ Q <= AF)

optimize!(model)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
LP has 3 rows; 6 cols; 18 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 5e+00]
  Cost    [3e+01, 6e+01]
  Bound   [5e+01, 9e+01]
  RHS     [3e+02, 5e+02]
Presolving model
3 rows, 6 cols, 18 nonzeros 0s
3 rows, 6 cols, 18 nonzeros 0s
Presolve reductions: rows 3(-0); columns 6(-0); nonzeros 18(-0) - Not reduced
Problem not reduced by presolve: solving the LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
          4     8.1633333333e+03 Pr: 0(0) 0.0s

Model status        : Optimal
Simplex   iterations: 4
Objective value     :  8.1633333333e+03
P-D objective error :  5.5702674506e-17
HiGHS run time      :          0.00


In [4]:
Q_ = value(Q)
M ⋅ Q_ == AM, A ⋅ Q_ == AA, F ⋅ Q_ == AF


(false, true, true)

| $Q_1$ | $Q_2$ | $Q_3$ | $Q_4$ | $Q_5$ | $Q_6$ |
| ----- | ----- | ----- | ----- | ----- | ----- |
| 0     | 7.5   | 14.17 | 60    | 75    | 0.83  |

| Resource  | Used (h) | Available (h) | Slack (h) | Active |
|-----------|----------|---------------|-----------|--------|
| Machining | 520      | 520           | 0         | Active |
| Assembly  | 480      | 480           | 0         | Active |
| Finishing | 300      | 300           | 0         | Active |

# Operational production rules

In reality, producing a product requires a setup operation. A fixed setup cost is incurred whenever
the product is produced, independently of the quantity produced. Moreover, once production of
a product is launched, the production run must reach a minimum size.

| Product | Fixed setup cost (€) | Minimum production if activated |
| ------- | -------------------- | ------------------------------- |
| P1      | 400                  | 20                              |
| P2      | 450                  | 20                              |
| P3      | 500                  | 25                              |
| P4      | 700                  | 15                              |
| P5      | 500                  | 20                              |
| P6      | 600                  | 10                              |

The weekly profit is now the contribution margin generated by production minus the setup costs
of the products that are activated. There is also one technological dependency:
Product P4 may be produced only during a week in which product P1 is also produced.

In [5]:
FSC = [400, 450, 500, 700, 500, 600]
MINQ = [20, 20, 25, 15, 20, 10]

@variable(model, X[i = 1:N], Bin)

@objective(model, Max, U ⋅ Q - X ⋅ FSC)
@constraint(model, cX1, X .* MINQ - Q <= 0)
@constraint(model, cX2, 0 <= X .* MAX - Q)
@constraint(model, cDependance, X[1] >= X[4])

cDependance : X[1] - X[4] >= 0

In [6]:
optimize!(model)

value.(X), value.(Q), value.(U⋅Q - X⋅FSC)

MIP has 16 rows; 12 cols; 44 nonzeros; 6 integer variables (6 binary)
Coefficient ranges:
  Matrix  [1e+00, 9e+01]
  Cost    [3e+01, 7e+02]
  Bound   [1e+00, 9e+01]
  RHS     [3e+02, 5e+02]
Presolving model
16 rows, 12 cols, 44 nonzeros 0s
16 rows, 12 cols, 44 nonzeros 0s
Presolve reductions: rows 16(-0); columns 12(-0); nonzeros 44(-0) - Not reduced

Solving MIP model with:
   16 rows
   12 cols (6 binary, 0 integer, 0 implied int., 6 continuous, 0 domain fixed)
   44 nonzeros
   Thread count 6 (of 12 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective Bo

([0.0, 1.0, 0.0, 0.0, 1.0, 1.0], [0.0, 70.0, 0.0, 0.0, 70.0, 20.0], 6160.0)